In [ ]:
import fsspec
import pandas as pd

S3_REGION = "ap-northeast-1"
SO = {"client_kwargs": {"region_name": S3_REGION}}

SYMBOL = "BTCUSDT"
YEAR = 2024
MONTHS = [1, 2, 3, 4, 5, 6]  # <-- 6 mois

fs = fsspec.filesystem("s3", **SO)

def list_month_parts(symbol: str, year: int, month: int) -> list[str]:
    part_glob = (
        f"s3://tradebot-config-tokyo/data/stageA/"
        f"symbol={symbol}/year={year}/month={month:02d}/part-*.parquet"
    )
    _, _, paths = fsspec.get_fs_token_paths(part_glob, storage_options=SO)
    paths = sorted(paths)

    # IMPORTANT: remettre s3:// devant si absent
    def to_s3_url(p: str) -> str:
        return p if p.startswith("s3://") else f"s3://{p}"

    return [to_s3_url(p) for p in paths]

all_paths = []
for m in MONTHS:
    p = list_month_parts(SYMBOL, YEAR, m)
    print(f"month={m:02d} -> n_parts={len(p)}  example={p[:1]}")
    all_paths.extend(p)

all_paths = sorted(all_paths)

print("\nTOTAL n_parts:", len(all_paths))
print("example:", all_paths[:3])

In [ ]:
MIN_COLS = [
    "id_t",
    "label_A",
    "label_A_exit_reason",
    "label_A_pnl_net_bps",
    "label_A_exit_t_sec",
    "audit_spread_bps_entry",
    "audit_cost_bps",
    "audit_R_bps",
    "audit_tp_bps",
    "audit_sl_bps",
    "audit_rr_min",
    "audit_cost_R",
    "audit_p_thr_ev0",
    "cfg_horizon_sec",
    "cfg_decision_step_sec",
    "cfg_k_confirm_sec",
]

def read_parts_stream(paths, columns):
    for p in paths:
        yield pd.read_parquet(
            p,
            storage_options=SO,
            columns=columns,
            engine="pyarrow",
        )

dfs = []
n_rows = 0

for df_part in read_parts_stream(all_paths, MIN_COLS):
    n_rows += len(df_part)
    dfs.append(df_part)

df = pd.concat(dfs, ignore_index=True)

print("rows:", n_rows)
print("shape:", df.shape)
df.head()

In [ ]:
# --- 3)  
import numpy as np
import pandas as pd

print("shape:", df.shape)

# --- dtypes attendus (souple, mais utile)
EXPECTED_DTYPES = {
    "id_t": "datetime64[ns, UTC]",
    "label_A": "int8",
    "label_A_exit_t_sec": "int16",
    "audit_cost_bps": "float32",
    "audit_R_bps": "float32",
    "audit_tp_bps": "float32",
    "audit_sl_bps": "float32",
    "audit_cost_R": "float32",
    "audit_p_thr_ev0": "float32",
}

print("\n--- dtypes ---")
print(df.dtypes)

# --- checks simples
print("\n--- duplicates id_t ---")
dup = df["id_t"].duplicated().mean()
print("duplicate ratio:", dup)

print("\n--- monotonicity (si trié) ---")
df_sorted = df.sort_values("id_t")
mono = df_sorted["id_t"].is_monotonic_increasing
print("id_t monotonic increasing after sort:", mono)

print("\n--- NA ratios ---")
na = df.isna().mean().sort_values(ascending=False)
print(na.head(10))

# --- ranges sanity
print("\n--- ranges sanity ---")
print("label_A unique:", sorted(df["label_A"].dropna().unique().tolist()))
print("exit_t_sec min/max:", int(df["label_A_exit_t_sec"].min()), int(df["label_A_exit_t_sec"].max()))
print("p_thr_ev0 min/max:", float(df["audit_p_thr_ev0"].min()), float(df["audit_p_thr_ev0"].max()))

# hard asserts (si tu veux que ça fail vite)
assert df["label_A"].isin([0,1]).all()
assert df["audit_p_thr_ev0"].between(0.0, 1.0).all()
assert df["label_A_exit_t_sec"].between(-1, 120).all()

In [ ]:
# --- 5) df["month"] = df["id_t"].dt.to_period("M").astype(str)

# global
print("N rows:", len(df))
print("label_A mean:", df["label_A"].mean())

er = df["label_A_exit_reason"].fillna("NA")
dist = er.value_counts()
dist_pct = (dist / len(df) * 100).round(3)
out = pd.DataFrame({"count": dist, "pct": dist_pct})
print("\nexit_reason distribution:\n", out)

# par mois
g = df.groupby("month").agg(
    n=("label_A","size"),
    label_mean=("label_A","mean"),
    tp=("label_A", "sum"),
    nofill=("label_A_exit_reason", lambda s: (s=="NOFILL").sum()),
    time=("label_A_exit_reason", lambda s: (s=="TIME").sum()),
    cf=("label_A_exit_reason", lambda s: (s=="CONFIRM_FAIL").sum()),
)
g["tp_rate"] = g["tp"]/g["n"]
g["nofill_rate"] = g["nofill"]/g["n"]
g["time_rate"] = g["time"]/g["n"]
g["cf_rate"] = g["cf"]/g["n"]

print("\n--- monthly summary ---")
display(g.sort_index())

In [ ]:
# --- 4) 
er = df["label_A_exit_reason"].fillna("NA")

filled_loose = ~er.isin(["NOFILL"])
filled_strict = ~er.isin(["NOFILL","CONFIRM_FAIL"])

print("filled share (loose, not NOFILL):", float(filled_loose.mean()))
print("filled share (strict, not NOFILL/CONFIRM_FAIL):", float(filled_strict.mean()))

df_f = df.loc[filled_loose].copy()
dist_f = df_f["label_A_exit_reason"].value_counts(normalize=True).sort_values(ascending=False)
print("\nexit_reason (filled only):")
print(dist_f)

# check logique: TP => label_A==1, SL/TIME => label_A==0
bad_tp = df[(er.isin(["TP_LONG","TP_SHORT"])) & (df["label_A"] != 1)]
bad_non_tp = df[(~er.isin(["TP_LONG","TP_SHORT"])) & (df["label_A"] != 0)]

print("\nBad TP rows (should be 0):", len(bad_tp))
print("Bad non-TP rows (should be 0):", len(bad_non_tp))

assert len(bad_tp) == 0
assert len(bad_non_tp) == 0

In [ ]:
# --- 6) 
def q(s, ps=(0.01,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.99)):
    return s.quantile(list(ps))

print("audit_tp_bps quantiles:\n", q(df["audit_tp_bps"]))
print("\naudit_sl_bps quantiles:\n", q(df["audit_sl_bps"]))
print("\naudit_cost_bps quantiles:\n", q(df["audit_cost_bps"]))
print("\naudit_cost_R quantiles:\n", q(df["audit_cost_R"]))
print("\naudit_p_thr_ev0 quantiles:\n", q(df["audit_p_thr_ev0"]))

# corr rapide
corr = df[["audit_cost_R","audit_p_thr_ev0","audit_tp_bps","audit_sl_bps","audit_cost_bps"]].corr(numeric_only=True)
print("\n--- correlations ---")
display(corr)

In [ ]:
# --- 7) 
tp_mask = df["label_A"] == 1
df_tp = df.loc[tp_mask].copy()

print("TP count:", len(df_tp))
if len(df_tp) > 0:
    print("\nTP pnl_net_bps quantiles:\n", df_tp["label_A_pnl_net_bps"].quantile([0.01,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.99]))

    # outliers (top 10)
    print("\nTop 10 TP pnl:")
    display(df_tp.sort_values("label_A_pnl_net_bps", ascending=False).head(10)[
        ["id_t","label_A_exit_reason","label_A_pnl_net_bps","audit_tp_bps","audit_cost_bps","audit_R_bps","audit_cost_R"]
    ])

# TP counts par mois
tp_month = df_tp["id_t"].dt.to_period("M").astype(str).value_counts().sort_index()
print("\nTP counts per month:")
display(tp_month)

In [1]:
# === Cell unique: sanity check prices_1s (id_t, mid) sur un ou plusieurs jours ===
# Vérifie:
# - existence des fichiers
# - schéma (id_t int64, mid float)
# - mid > 0, pas de NaN/inf
# - id_t monotone + stats de gaps (diff)
# - couverture temporelle et cadence ~1s

import numpy as np
import pandas as pd
import s3fs

# -------------------
# CONFIG
# -------------------
PRICES_ROOT = "s3://tradebot-config-tokyo/data/stageA/prices_1s"
SYMBOL = "BTCUSDT"

# mets 1 ou plusieurs dates
DATES = ["2024-01-01"]  # ex: ["2024-01-01","2024-01-02"]

# limite de lignes à lire (None = tout)
MAX_ROWS = None  # ex 2_000_000
# -------------------

fs = s3fs.S3FileSystem()

def list_parts(date_str: str):
    prefix = f"{PRICES_ROOT.rstrip('/')}/symbol={SYMBOL}/date={date_str}"
    pattern = prefix.replace("s3://", "") + "/part-*.parquet"
    paths = [f"s3://{p}" for p in fs.glob(pattern)]
    if not paths:
        raise FileNotFoundError(f"No part-*.parquet found under {prefix}")
    return sorted(paths)

def load_day(date_str: str):
    paths = list_parts(date_str)
    dfs = []
    for p in paths:
        df = pd.read_parquet(p, columns=["id_t","mid"], engine="pyarrow")
        dfs.append(df)
    out = pd.concat(dfs, ignore_index=True)
    if MAX_ROWS is not None and len(out) > int(MAX_ROWS):
        out = out.iloc[:int(MAX_ROWS)].copy()
    return out

def summarize_gaps(id_t: np.ndarray):
    d = np.diff(id_t.astype(np.int64))
    if d.size == 0:
        return {"n": 0}
    return {
        "n": int(d.size),
        "p50": float(np.quantile(d, 0.50)),
        "p95": float(np.quantile(d, 0.95)),
        "p99": float(np.quantile(d, 0.99)),
        "max": float(np.max(d)),
        "frac_eq_1": float(np.mean(d == 1)),
        "frac_gt_1": float(np.mean(d > 1)),
        "n_gaps_gt_1": int(np.sum(d > 1)),
    }

rows = []
for d in DATES:
    df = load_day(d)

    # --- schema checks ---
    if not {"id_t","mid"}.issubset(df.columns):
        raise RuntimeError(f"[{d}] missing columns. got={df.columns.tolist()}")
    if df["id_t"].dtype != np.int64 and str(df["id_t"].dtype) != "int64":
        raise RuntimeError(f"[{d}] id_t dtype must be int64. got={df['id_t'].dtype}")
    if not (np.issubdtype(df["mid"].dtype, np.floating)):
        raise RuntimeError(f"[{d}] mid dtype must be float. got={df['mid'].dtype}")

    # --- value checks ---
    mid = df["mid"].to_numpy(dtype=np.float64, copy=False)
    if not np.isfinite(mid).all():
        raise RuntimeError(f"[{d}] mid has NaN/inf")
    if (mid <= 0).any():
        raise RuntimeError(f"[{d}] mid has <=0 values")

    id_t = df["id_t"].to_numpy(dtype=np.int64, copy=False)
    if not np.isfinite(id_t).all():
        raise RuntimeError(f"[{d}] id_t has NaN/inf (unexpected)")
    # monotonic (non-decreasing) + ideally strictly increasing after sort
    if not np.all(id_t[1:] >= id_t[:-1]):
        # si pas trié, on trie et on re-check
        df = df.sort_values("id_t").reset_index(drop=True)
        id_t = df["id_t"].to_numpy(dtype=np.int64, copy=False)
        if not np.all(id_t[1:] >= id_t[:-1]):
            raise RuntimeError(f"[{d}] id_t not monotone even after sort")

    # duplicates
    n_dup = int(len(id_t) - len(np.unique(id_t)))

    gaps = summarize_gaps(id_t)

    # cadence approximative
    span = int(id_t[-1] - id_t[0]) if len(id_t) else 0
    expected = span + 1 if span >= 0 else 0
    coverage = float(len(id_t) / expected) if expected > 0 else 0.0

    rows.append({
        "date": d,
        "n_rows": int(len(df)),
        "id_t_min": int(id_t[0]) if len(id_t) else None,
        "id_t_max": int(id_t[-1]) if len(id_t) else None,
        "span_sec": int(span),
        "coverage_vs_full_1s": coverage,  # ~1.0 si tu as (quasi) toutes les secondes
        "n_duplicates": n_dup,
        "mid_mean": float(np.mean(mid)) if len(mid) else None,
        "mid_p01": float(np.quantile(mid, 0.01)) if len(mid) else None,
        "mid_p50": float(np.quantile(mid, 0.50)) if len(mid) else None,
        "mid_p99": float(np.quantile(mid, 0.99)) if len(mid) else None,
        "gap_p50": gaps.get("p50", None),
        "gap_p95": gaps.get("p95", None),
        "gap_p99": gaps.get("p99", None),
        "gap_max": gaps.get("max", None),
        "frac_gap_eq_1": gaps.get("frac_eq_1", None),
        "frac_gap_gt_1": gaps.get("frac_gt_1", None),
        "n_gaps_gt_1": gaps.get("n_gaps_gt_1", None),
    })

summary = pd.DataFrame(rows)
display(summary)

print("\nHeuristiques:")
print("- coverage_vs_full_1s proche de 1.0 => tu as (quasi) toutes les secondes.")
print("- frac_gap_eq_1 proche de 1.0 => cadence ~1s régulière.")
print("- n_duplicates devrait être 0 (ou très faible si ingestion bizarre).")

,date,n_rows,id_t_min,id_t_max,span_sec,coverage_vs_full_1s,n_duplicates,mid_mean,mid_p01,mid_p50,mid_p99,gap_p50,gap_p95,gap_p99,gap_max,frac_gap_eq_1,frac_gap_gt_1,n_gaps_gt_1
0,2024-01-01,86400,1704067200,1704153599,86399,1.0,0,42840.476978,42273.851562,42699.949219,44136.649453,1.0,1.0,1.0,1.0,1.0,0.0,0



Heuristiques:
- coverage_vs_full_1s proche de 1.0 => tu as (quasi) toutes les secondes.
- frac_gap_eq_1 proche de 1.0 => cadence ~1s régulière.
- n_duplicates devrait être 0 (ou très faible si ingestion bizarre).
